# Ingestion Patterns

Notebook for **Chapter 4** of the *de* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/de/ch04.html

Tip: change a number, predict the output, then run the cell again.

## 4.1 Full Load vs Incremental Load and the Watermark

In [ ]:
import sqlite3

SOURCE = [{"id": i, "amount": 100 + i, "updated_at": f"2024-05-{(i % 5) + 1:02d}T10:00:00"} for i in range(1, 26)]

def api_get(since, page, page_size=10):
    """Fake: GET /orders?updated_after=..&page=..  (sorted by updated_at, id)"""
    data = sorted([r for r in SOURCE if r["updated_at"] > since], key=lambda r: (r["updated_at"], r["id"]))
    chunk = data[page * page_size:(page + 1) * page_size]
    return {"items": chunk, "has_more": (page + 1) * page_size < len(data)}

db = sqlite3.connect(":memory:")
db.executescript("""
CREATE TABLE orders(id INTEGER PRIMARY KEY, amount REAL, updated_at TEXT);
CREATE TABLE etl_state(pipeline TEXT PRIMARY KEY, watermark TEXT);
INSERT INTO etl_state VALUES('orders','1970-01-01T00:00:00');
""")

def run_once():
    wm = db.execute("SELECT watermark FROM etl_state WHERE pipeline='orders'").fetchone()[0]
    page, fetched, new_wm = 0, 0, wm
    while True:
        resp = api_get(wm, page)
        for r in resp["items"]:
            db.execute("""INSERT INTO orders VALUES(:id,:amount,:updated_at)
                          ON CONFLICT(id) DO UPDATE SET amount=excluded.amount, updated_at=excluded.updated_at""", r)
            new_wm = max(new_wm, r["updated_at"]); fetched += 1
        if not resp["has_more"]: break
        page += 1
    db.execute("UPDATE etl_state SET watermark=? WHERE pipeline='orders'", (new_wm,))
    db.commit()
    total = db.execute("SELECT COUNT(*) FROM orders").fetchone()[0]
    print(f"run: wm_in={wm} fetched={fetched} pages={page+1} wm_out={new_wm} rows_in_table={total}")

run_once()                       # first load
run_once()                       # nothing new
SOURCE[2]["amount"] = 999; SOURCE[2]["updated_at"] = "2024-06-01T09:00:00"        # an update
SOURCE.append({"id": 26, "amount": 126, "updated_at": "2024-06-01T09:30:00"})     # an insert
run_once()                       # picks up the 2 changes
db.execute("UPDATE etl_state SET watermark='1970-01-01T00:00:00'"); db.commit()    # a backfill: reset the watermark
run_once()
print(db.execute("SELECT id, amount, updated_at FROM orders WHERE id IN (3,26)").fetchall())
print("duplicates:", db.execute("SELECT COUNT(*)-COUNT(DISTINCT id) FROM orders").fetchone()[0])

In [ ]:
SOURCE.append({"id": 27, "amount": 1, "updated_at": "2024-06-01T09:30:00"})   # same timestamp as id 26
run_once()
print("id 27 present?", db.execute("SELECT COUNT(*) FROM orders WHERE id=27").fetchone()[0])

run: wm_in=2024-06-01T09:30:00 fetched=0 pages=1 wm_out=2024-06-01T09:30:00 rows_in_table=26
id 27 present after run with '>' watermark? 0


## 4.2 APIs, Pagination, Rate Limits, Retries, Files and Webhooks

In [ ]:
data = [{"id": i} for i in range(10, 0, -1)]      # newest first: ids 10..1
def offset_page(offset, limit): return data[offset:offset+limit]
def cursor_page(after_id, limit):                 # rows with id < after_id
    return [r for r in data if after_id is None or r["id"] < after_id][:limit]
seen = []; page1 = offset_page(0, 4); seen += [r["id"] for r in page1]
data.insert(0, {"id": 11})                        # a new row arrives
page2 = offset_page(4, 4); seen += [r["id"] for r in page2]
print("offset paging saw:", seen, "-> duplicate:", [x for x in set(seen) if seen.count(x) > 1])
data.pop(0)
seen = []; last = None
for step in range(3):
    pg = cursor_page(last, 4)
    if step == 1: data.insert(0, {"id": 11})
    if not pg: break
    seen += [r["id"] for r in pg]; last = pg[-1]["id"]
print("cursor paging saw:", seen, "-> duplicates:", len(seen) - len(set(seen)))

offset paging saw: [10, 9, 8, 7, 7, 6, 5, 4] -> duplicate: [7]
cursor paging saw: [10, 9, 8, 7, 6, 5, 4, 3, 2, 1] -> duplicates: 0


In [ ]:
import random
random.seed(11)
def flaky():
    if random.random() < 0.6: raise ConnectionError("503 service unavailable")
    return "ok"
def with_retry(fn, max_tries=6, base=1.0, cap=30.0):
    waited = 0.0
    for attempt in range(1, max_tries + 1):
        try: return fn(), attempt, waited
        except ConnectionError as e:
            if attempt == max_tries: raise
            delay = min(cap, base * 2 ** (attempt - 1)); jitter = random.uniform(0, delay)
            print(f"  attempt {attempt} failed ({e}); sleep up to {delay:.0f}s -> chose {jitter:.2f}s"); waited += jitter
res, att, w = with_retry(flaky); print("result", res, "after", att, "attempts, total wait", round(w, 2), "s")
print("max waits without jitter for 5 failures:", [min(30, 2**k) for k in range(5)], "sum", sum(min(30, 2**k) for k in range(5)))

  attempt 1 failed (503 service unavailable); sleep up to 1s -> chose 0.56s
result ok after 2 attempts, total wait 0.56 s
max waits without jitter for 5 failures: [1, 2, 4, 8, 16] sum 31


## 4.3 Change Data Capture, Backfills and Late-Arriving Data

In [ ]:
import sqlite3
db = sqlite3.connect(":memory:")
db.execute("CREATE TABLE cust(id INTEGER PRIMARY KEY, name TEXT, city TEXT)")
log = [(1,"c",1,{"name":"Asha","city":"Pune"}), (2,"c",2,{"name":"Ravi","city":"Delhi"}),
       (3,"u",1,{"city":"Mumbai"}), (4,"d",2,None),
       (5,"c",3,{"name":"Meena","city":"Chennai"}), (6,"u",3,{"city":"Kochi"})]
def apply(ev):
    lsn, op, k, d = ev
    if op == "c": db.execute("INSERT OR REPLACE INTO cust VALUES(?,?,?)", (k, d["name"], d["city"]))
    elif op == "u":
        for c, v in d.items(): db.execute(f"UPDATE cust SET {c}=? WHERE id=?", (v, k))
    else: db.execute("DELETE FROM cust WHERE id=?", (k,))
for e in log: apply(e)
print(db.execute("SELECT * FROM cust ORDER BY id").fetchall())
for e in log[2:4]: apply(e)
print("after replaying events 3,4 again:", db.execute("SELECT * FROM cust ORDER BY id").fetchall())

[(1, 'Asha', 'Mumbai'), (3, 'Meena', 'Kochi')]
after replaying events 3,4 again: [(1, 'Asha', 'Mumbai'), (3, 'Meena', 'Kochi')]


## 4.4 Python for Data Engineering: Generators and Chunked Reads

In [ ]:
import csv, os, tempfile, tracemalloc
path = os.path.join(tempfile.mkdtemp(), "orders.csv")
with open(path, "w", newline="") as f:
    w = csv.writer(f); w.writerow(["order_id", "city", "amount"])
    for i in range(200_000):
        w.writerow([f"O{i}", ["Pune", "Delhi", "Mumbai", "Chennai"][i % 4], 100 + i % 900])

def read_all(p):                      # the truck: every row at once
    with open(p, newline="") as f:
        return list(csv.DictReader(f))

def read_lazy(p):                     # the conveyor belt: one row at a time
    with open(p, newline="") as f:
        for row in csv.DictReader(f):
            yield row

def measure(make_rows):
    tracemalloc.start()
    total = sum(int(r["amount"]) for r in make_rows(path))
    peak = tracemalloc.get_traced_memory()[1]
    tracemalloc.stop()
    return total, peak

t1, p1 = measure(read_all)
t2, p2 = measure(read_lazy)
print("same total:", t1 == t2, t1)
print("list peak is more than 50 times the generator peak:", p1 > 50 * p2)
print("generator peak under 1 MB:", p2 < 1_000_000)

same total: True 109830000
list peak is more than 50 times the generator peak: True
generator peak under 1 MB: True


In [ ]:
import numpy as np, pandas as pd, io
rng = np.random.default_rng(7)
n = 20_000
df = pd.DataFrame({"city": rng.choice(["Pune", "Delhi", "Mumbai"], n, p=[0.5, 0.3, 0.2]),
                   "amount": rng.integers(50, 500, n)})
text = df.to_csv(index=False)

sums, counts = {}, {}
for chunk in pd.read_csv(io.StringIO(text), chunksize=3_000):      # 7 chunks
    g = chunk.groupby("city")["amount"].agg(["sum", "count"])
    for city, row in g.iterrows():
        sums[city] = sums.get(city, 0) + int(row["sum"])
        counts[city] = counts.get(city, 0) + int(row["count"])
chunked = {c: round(sums[c] / counts[c], 3) for c in sorted(sums)}
full = df.groupby("city")["amount"].mean().round(3).to_dict()
print("chunked:", chunked)
print("full   :", dict(sorted(full.items())))
print("equal:", chunked == dict(sorted(full.items())))
print("rows seen:", sum(counts.values()))

chunked: {'Delhi': 276.74, 'Mumbai': 275.854, 'Pune': 275.814}
full   : {'Delhi': 276.74, 'Mumbai': 275.854, 'Pune': 275.814}
equal: True
rows seen: 20000


In [ ]:
import numpy as np, pandas as pd
rng = np.random.default_rng(1)
n = 1_000_000
df = pd.DataFrame({"qty": rng.integers(1, 20, n), "city": rng.choice(["Pune", "Delhi", "Mumbai"], n)})
print("qty int64 bytes:", df["qty"].memory_usage(index=False))
small = df["qty"].astype("int8")
print("qty int8  bytes:", small.memory_usage(index=False))
cat = df["city"].astype("category")
print("city as category is smaller than as text:", cat.memory_usage(deep=True, index=False) < df["city"].memory_usage(deep=True, index=False))
print("category codes dtype:", cat.cat.codes.dtype)

qty int64 bytes: 8000000
qty int8  bytes: 1000000
city as category is smaller than as text: True
category codes dtype: int8


## 4.5 Typed Records, Concurrency and Safe Retries

In [ ]:
from dataclasses import dataclass
from typing import Optional

ALLOWED = {"PAID", "SHIPPED", "REFUNDED"}

@dataclass(frozen=True)
class Order:
    order_id: str
    amount_paise: int
    status: str
    coupon: Optional[str] = None

    def __post_init__(self):
        if not isinstance(self.amount_paise, int) or self.amount_paise < 0:
            raise ValueError(f"amount_paise must be a non-negative int, got {self.amount_paise!r}")
        if self.status not in ALLOWED:
            raise ValueError(f"status {self.status!r} not in {sorted(ALLOWED)}")

raw = [{"order_id": "O1", "amount_paise": 25000, "status": "PAID"},
       {"order_id": "O2", "amount_paise": "1,250.50", "status": "PAID"},
       {"order_id": "O3", "amount_paise": 900, "status": "LOST"}]
good, bad = [], []
for r in raw:
    try: good.append(Order(**r))
    except ValueError as e: bad.append((r["order_id"], str(e)))
print("good:", [o.order_id for o in good])
for oid, why in bad: print("quarantine", oid, "->", why)

good: ['O1']
quarantine O2 -> amount_paise must be a non-negative int, got '1,250.50'
quarantine O3 -> status 'LOST' not in ['PAID', 'REFUNDED', 'SHIPPED']


In [ ]:
import asyncio, time

active = 0
max_active = 0

async def fetch_page(n, sem):
    global active, max_active
    async with sem:                       # at most 50 inside at once
        active += 1; max_active = max(max_active, active)
        await asyncio.sleep(0.01)         # stands in for the network wait
        active -= 1
        return {"page": n, "rows": 100}

async def main():
    sem = asyncio.Semaphore(50)
    t0 = time.perf_counter()
    pages = await asyncio.gather(*(fetch_page(n, sem) for n in range(1, 501)))
    return pages, time.perf_counter() - t0

pages, elapsed = asyncio.run(main())
print("pages:", len(pages), "rows:", sum(p["rows"] for p in pages))
print("in order:", [p["page"] for p in pages] == list(range(1, 501)))
print("max in flight:", max_active)
print("faster than sequential (5 s):", elapsed < 5.0)

pages: 500 rows: 50000
in order: True
max in flight: 50
faster than sequential (5 s): True


In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

def slow_lookup(customer_id):             # a blocking call, 0.05 s each
    time.sleep(0.05)
    return customer_id, f"C{customer_id:03d}"

ids = list(range(1, 21))
t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=5) as pool:
    results = list(pool.map(slow_lookup, ids))     # keeps the input order
elapsed = time.perf_counter() - t0
print(results[:3], "...", results[-1])
print("20 calls x 0.05 s = 1.0 s sequential; threaded under 0.6 s:", elapsed < 0.6)

[(1, 'C001'), (2, 'C002'), (3, 'C003')] ... (20, 'C020')
20 calls x 0.05 s = 1.0 s sequential; threaded under 0.6 s: True


## 4.6 An Idempotent Loader Function

In [ ]:
import sqlite3

db = sqlite3.connect(":memory:")
db.executescript("""
CREATE TABLE orders(order_id TEXT PRIMARY KEY, amount INTEGER, updated_at TEXT);
CREATE TABLE load_log(run_id TEXT PRIMARY KEY, rows_in INTEGER);
""")

def load_batch(db, run_id, rows, fail_after=None):
    with db:                                           # one transaction: commit or roll back
        for i, (oid, amount, ts) in enumerate(rows):
            if fail_after is not None and i == fail_after:
                raise RuntimeError("worker died mid-batch")
            db.execute("""INSERT INTO orders VALUES (?,?,?)
                          ON CONFLICT(order_id) DO UPDATE
                          SET amount = excluded.amount, updated_at = excluded.updated_at
                          WHERE excluded.updated_at >= orders.updated_at""", (oid, amount, ts))
        db.execute("INSERT OR REPLACE INTO load_log VALUES (?,?)", (run_id, len(rows)))

def state():
    return db.execute("SELECT order_id, amount FROM orders ORDER BY order_id").fetchall()

batch = [("O1", 100, "2024-06-01T10:00"), ("O2", 200, "2024-06-01T11:00"), ("O3", 300, "2024-06-01T12:00")]
load_batch(db, "run-1", batch)
print("after run 1:     ", state())
load_batch(db, "run-1", batch)                         # retry of the same run
print("after retry:     ", state())
load_batch(db, "run-2", [("O2", 999, "2024-06-01T09:00")])   # a stale, older version of O2
print("after stale row: ", state())
load_batch(db, "run-3", [("O2", 250, "2024-06-02T08:00")])   # a genuine update
print("after update:    ", state())
try:
    load_batch(db, "run-4", [("O4", 400, "2024-06-02T09:00"), ("O5", 500, "2024-06-02T09:05")], fail_after=1)
except RuntimeError as e:
    print("run 4 failed:", e)
print("after failed run:", state())
print("log:", db.execute("SELECT run_id, rows_in FROM load_log ORDER BY run_id").fetchall())

after run 1:      [('O1', 100), ('O2', 200), ('O3', 300)]
after retry:      [('O1', 100), ('O2', 200), ('O3', 300)]
after stale row:  [('O1', 100), ('O2', 200), ('O3', 300)]
after update:     [('O1', 100), ('O2', 250), ('O3', 300)]
run 4 failed: worker died mid-batch
after failed run: [('O1', 100), ('O2', 250), ('O3', 300)]
log: [('run-1', 3), ('run-2', 1), ('run-3', 1)]


## 4.7 Log-Based CDC in Depth: Events, Tombstones, Snapshots and the Outbox

In [ ]:
events = [  # (log position, op, key, after-value)
    (1, "c", "O1", 100),
    (2, "c", "O2", 200),
    (3, "u", "O1", 150),
    (4, "d", "O2", None),
    (5, "c", "O3", 300),
]

def apply(state, ev, keep_marker=True):
    pos, op, key, val = ev
    if keep_marker and key in state and pos <= state[key][1]:
        return                                      # already applied or older: skip
    if op == "d":
        if keep_marker: state[key] = (None, pos)   # remember the delete and its position
        else: state.pop(key, None)                  # naive: forget the key
    else:
        state[key] = (val, pos)

def live(state):
    return {k: v[0] for k, v in sorted(state.items()) if v[0] is not None}

s = {}
for ev in events: apply(s, ev)
print("first pass :", live(s))
for ev in events: apply(s, ev)                      # full replay
print("after replay:", live(s))
late = (2, "c", "O2", 200)                          # an old event arrives late
apply(s, late)
print("late old event, guarded:", live(s))

naive = {}
for ev in events: apply(naive, ev, keep_marker=False)
apply(naive, late, keep_marker=False)
print("late old event, naive  :", live(naive))

first pass : {'O1': 150, 'O3': 300}
after replay: {'O1': 150, 'O3': 300}
late old event, guarded: {'O1': 150, 'O3': 300}
late old event, naive  : {'O1': 150, 'O2': 200, 'O3': 300}


In [ ]:
snapshot_pos = 100
snapshot = {"O1": 150, "O2": 200, "O3": 300}                 # table contents as of position 100
log = [(98, "u", "O1", 150),                                   # already inside the snapshot
       (101, "u", "O2", 250),                                  # after the snapshot
       (102, "c", "O4", 400),
       (103, "d", "O3", None)]

state = {k: (v, snapshot_pos) for k, v in snapshot.items()}
for pos, op, key, val in log:
    if key in state and pos <= state[key][1]:
        print(f"skip   pos {pos} {op} {key} (inside snapshot)")
        continue
    state[key] = (None, pos) if op == "d" else (val, pos)
    print(f"apply  pos {pos} {op} {key}")
print({k: v[0] for k, v in sorted(state.items()) if v[0] is not None})

skip   pos 98 u O1 (inside snapshot)
apply  pos 101 u O2
apply  pos 102 c O4
apply  pos 103 d O3
{'O1': 150, 'O2': 250, 'O4': 400}


In [ ]:
import sqlite3, json
db = sqlite3.connect(":memory:")
db.executescript("""
CREATE TABLE orders(order_id TEXT PRIMARY KEY, amount INTEGER);
CREATE TABLE outbox(event_id INTEGER PRIMARY KEY AUTOINCREMENT, payload TEXT, sent INTEGER DEFAULT 0);
""")

def place_order(order_id, amount, crash=False):
    try:
        with db:                                   # one transaction for both writes
            db.execute("INSERT INTO orders VALUES (?,?)", (order_id, amount))
            db.execute("INSERT INTO outbox(payload) VALUES (?)", (json.dumps({"order_id": order_id, "amount": amount}),))
            if crash: raise RuntimeError("crash before commit")
    except RuntimeError: pass

place_order("O1", 100)
place_order("O2", 200, crash=True)                 # neither row is written
print("orders:", db.execute("SELECT COUNT(*) FROM orders").fetchone()[0],
      "outbox:", db.execute("SELECT COUNT(*) FROM outbox").fetchone()[0])

published = []                                     # stands in for Kafka
def relay(crash_before_mark=False):
    for eid, payload in db.execute("SELECT event_id, payload FROM outbox WHERE sent = 0").fetchall():
        published.append((eid, payload))           # publish first
        if crash_before_mark: return               # crash: the row is not marked as sent
        db.execute("UPDATE outbox SET sent = 1 WHERE event_id = ?", (eid,)); db.commit()

relay(crash_before_mark=True)
relay()                                            # restart: publishes the same event again
print("published:", [e for e, _ in published])
seen, processed = set(), []
for eid, payload in published:                     # consumer deduplicates by event id
    if eid in seen: continue
    seen.add(eid); processed.append(eid)
print("processed once:", processed)

orders: 1 outbox: 1
published: [1, 1]
processed once: [1]


## Chapter recap

## Practice set

**1.** (easy) 95 rows, page size 20. How many pages? **Answer:** ⌈95 / 20⌉ = 5.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/de/ch04.html